In [1]:
%pip install -q -U pandas requests tqdm
from datetime import datetime, timezone
import time
import pandas as pd
import requests
from tqdm.auto import tqdm

NETID = 'aberard'
PROJECTS = {
    'benfulcher_hctsa': 'benfulcher/hctsa',
    'juliaopt_jump.jl': 'JuliaOpt/JuMP.jl',
    'fredros_pypmc': 'fredRos/pypmc',
    'rpng_open_vins': 'rpng/open_vins',
    'cp2k_dbcsr': 'cp2k/dbcsr',
    'pbs-assess_gftrends': 'pbs-assess/gftrends',
    'juliamolsim_dftk.jl': 'JuliaMolSim/DFTK.jl',
    'eclipse_sumo': 'eclipse/sumo',
    'iqss_dataverse': 'IQSS/dataverse',
    'ecp-copa_cabana': 'ECP-COPA/Cabana',
}
BATCH_SIZE = 10  # below the assignment's limit of 50
WAIT_SECONDS = 1
WOC_API = 'https://worldofcode.org/api'

def woc_get(path, params=None, retries=4):
    for attempt in range(retries):
        response = requests.get(f'{WOC_API}{path}', params=params, timeout=90)
        if response.status_code == 429 and attempt < retries - 1:
            time.sleep(int(response.headers.get('Retry-After', 10)))
            continue
        response.raise_for_status()
        return response.json()

Note: you may need to restart the kernel to use updated packages.


/home/fived/projects/woc/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
github_rows = []
for project_wocid, repo in PROJECTS.items():
    response = requests.get(f'https://api.github.com/repos/{repo}', headers={'Accept': 'application/vnd.github+json'}, timeout=30)
    response.raise_for_status()
    data = response.json()
    latest = requests.get(f'https://api.github.com/repos/{repo}/commits', params={'per_page': 1}, headers={'Accept': 'application/vnd.github+json'}, timeout=30)
    latest.raise_for_status()
    github_rows.append({
        'project_wocid': project_wocid,
        'github_repository': repo,
        'stars': data['stargazers_count'],
        'forks': data['forks_count'],
        'last_commit_date': latest.json()[0]['commit']['committer']['date'][:10],
    })
github_df = pd.DataFrame(github_rows)
display(github_df)

,project_wocid,github_repository,stars,forks,last_commit_date
0,benfulcher_hctsa,benfulcher/hctsa,860,367,2026-10-05
1,juliaopt_jump.jl,JuliaOpt/JuMP.jl,2482,429,2026-10-07
2,fredros_pypmc,fredRos/pypmc,50,13,2026-04-07
3,rpng_open_vins,rpng/open_vins,3132,939,2025-11-30
4,cp2k_dbcsr,cp2k/dbcsr,157,60,2026-10-08
5,pbs-assess_gftrends,pbs-assess/gftrends,0,0,2026-02-26
6,juliamolsim_dftk.jl,JuliaMolSim/DFTK.jl,540,111,2026-10-09
7,eclipse_sumo,eclipse/sumo,4207,1763,2026-10-10
8,iqss_dataverse,IQSS/dataverse,1076,569,2026-10-09
9,ecp-copa_cabana,ECP-COPA/Cabana,284,67,2026-09-16


In [3]:
project_commits = {}
for project_wocid in tqdm(PROJECTS, desc='WoC project-to-commit lookups'):
    project_commits[project_wocid] = woc_get(f'/lookup/map/p2c/{project_wocid}')['data']
    time.sleep(WAIT_SECONDS)

sha_df = pd.DataFrame(
    [(project, sha) for project, shas in project_commits.items() for sha in shas],
    columns=['project_wocid', 'commit_sha1'],
)
sha_df.to_csv(f'{NETID}_commit_sha1s.csv', index=False, sep=';')
display(sha_df.groupby('project_wocid').size().rename('number_of_commits').to_frame())

WoC project-to-commit lookups: 100%|██████████| 10/10 [00:17<00:00,  1.70s/it]


,number_of_commits
project_wocid,
benfulcher_hctsa,3783
cp2k_dbcsr,6631
eclipse_sumo,77331
ecp-copa_cabana,4470
fredros_pypmc,487
iqss_dataverse,42973
juliamolsim_dftk.jl,7848
juliaopt_jump.jl,12358
pbs-assess_gftrends,441


In [4]:

rows = []
errors = []
for project_wocid, shas in project_commits.items():
    for start in tqdm(range(0, len(shas), BATCH_SIZE), desc=project_wocid):
        batch = shas[start:start + BATCH_SIZE]
        response = woc_get('/lookup/map/commit.tch', params=[('q', sha) for sha in batch])
        result = response['data']
        batch_errors = response.get('errors', {})
        errors.extend((project_wocid, sha, error) for sha, error in batch_errors.items())
        for sha, values in result.items():
            commit = values[0]
            rows.append({
                'project_wocid': project_wocid,
                'commit_sha1': sha,
                'author': commit[2][0],
                'time': int(commit[2][1]),
                'commit message': commit[4],
            })
        time.sleep(WAIT_SECONDS)

ecp-copa_cabana: 100%|██████████| 447/447 [10:49<00:00,  1.45s/it]


In [5]:

summary_df = pd.DataFrame(rows, columns=['project_wocid', 'commit_sha1', 'author', 'time', 'commit message'])
summary_path = f'{NETID}_project_summary.csv'
summary_df.to_csv(summary_path, index=False, sep=';')

assert list(summary_df.columns) == ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']
assert not summary_df.empty, 'No commit rows were retrieved.'
assert summary_df['project_wocid'].nunique() == len(PROJECTS), 'At least one assigned project has no rows.'
print(f'Wrote {len(summary_df):,} rows to {summary_path}')
display(summary_df.head())

Wrote 156,991 rows to aberard_project_summary.csv


,project_wocid,commit_sha1,author,time,commit message
0,benfulcher_hctsa,000279bf51c5d92b79e6a3a7d470857d5bd473f5,Ben Fulcher <ben.d.fulcher@gmail.com>,1628146803,Support for custom annotations in TS_FeatureSu...
1,benfulcher_hctsa,000bd184adca5a5a7db5e14df23a22b818536dc6,Ben Fulcher <ben.fulcher@gmail.com>,1378821153,A few fixes here and there. Changed DV_dynsys_...
2,benfulcher_hctsa,0010375160c8434599743992b75b32818a358887,Ben Fulcher <ben.d.fulcher@gmail.com>,1440564091,Added fast implementation of visibility graph ...
3,benfulcher_hctsa,00311d91f87cc13464c76e2885ea1b26215f9ffb,GitHub Action <action@github.com>,1738278259,ghrs: vc agg 01-30-2304-ED44 for benfulcher/hc...
4,benfulcher_hctsa,004348617eef519b08c24802632e8d579def2878,Ben Fulcher <ben.fulcher@gmail.com>,1370535242,Made a bunch of changes in setting up the new ...


In [6]:
woc_stats = (summary_df.groupby('project_wocid', as_index=False)
    .agg(number_of_commits=('commit_sha1', 'size'),
         number_of_authors=('author', 'nunique'),
         max_time=('time', 'max'),
         min_time=('time', 'min')))
woc_stats['max_time_utc'] = pd.to_datetime(woc_stats['max_time'], unit='s', utc=True)
woc_stats['min_time_utc'] = pd.to_datetime(woc_stats['min_time'], unit='s', utc=True)
part1_report = woc_stats.merge(github_df, on='project_wocid', how='left')
part1_report.to_csv(f'{NETID}_part1_metadata.csv', index=False, sep=';')
display(part1_report)

,project_wocid,number_of_commits,number_of_authors,max_time,min_time,max_time_utc,min_time_utc,github_repository,stars,forks,last_commit_date
0,benfulcher_hctsa,3771,33,1762556662,1368696533,2025-11-07 23:04:22+00:00,2013-05-16 09:28:53+00:00,benfulcher/hctsa,860,367,2026-10-05
1,cp2k_dbcsr,6611,101,1762204068,993574001,2025-11-03 21:07:48+00:00,2001-06-26 16:46:41+00:00,cp2k/dbcsr,157,60,2026-10-08
2,eclipse_sumo,77293,356,1762285770,1018250481,2025-11-04 19:49:30+00:00,2002-04-08 07:21:21+00:00,eclipse/sumo,4207,1763,2026-10-10
3,ecp-copa_cabana,4456,94,1762280143,1515598380,2025-11-04 18:15:43+00:00,2018-01-10 15:33:00+00:00,ECP-COPA/Cabana,284,67,2026-09-16
4,fredros_pypmc,484,21,1762209048,1383226130,2025-11-03 22:30:48+00:00,2013-10-31 13:28:50+00:00,fredRos/pypmc,50,13,2026-04-07
5,iqss_dataverse,42924,537,1761751353,1383332803,2025-10-29 15:22:33+00:00,2013-11-01 19:06:43+00:00,IQSS/dataverse,1076,569,2026-10-09
6,juliamolsim_dftk.jl,7848,110,1776361883,1555434953,2026-04-16 17:51:23+00:00,2019-04-16 17:15:53+00:00,JuliaMolSim/DFTK.jl,540,111,2026-10-09
7,juliaopt_jump.jl,12286,295,1761691895,1349829286,2025-10-28 22:51:35+00:00,2012-10-10 00:34:46+00:00,JuliaOpt/JuMP.jl,2482,429,2026-10-07
8,pbs-assess_gftrends,441,7,1760638172,1604165393,2025-10-16 18:09:32+00:00,2020-10-31 17:29:53+00:00,pbs-assess/gftrends,0,0,2026-02-26
9,rpng_open_vins,877,58,1754832746,1547825046,2025-08-10 13:32:26+00:00,2019-01-18 15:24:06+00:00,rpng/open_vins,3132,939,2025-11-30


In [7]:
with open("aberard_project_summary.csv", "r", encoding="utf-8-sig") as f:
    for i in range(10):
        line = f.readline()
        if not line:
            break
        print(f"Line {i+1}: {line.rstrip()}")

Line 1: project_wocid;commit_sha1;author;time;commit message
Line 2: benfulcher_hctsa;000279bf51c5d92b79e6a3a7d470857d5bd473f5;Ben Fulcher <ben.d.fulcher@gmail.com>;1628146803;"Support for custom annotations in TS_FeatureSummary
Line 3: "
Line 4: benfulcher_hctsa;000bd184adca5a5a7db5e14df23a22b818536dc6;Ben Fulcher <ben.fulcher@gmail.com>;1378821153;"A few fixes here and there. Changed DV_dynsys_dbwell names to PH_ForcePotential.
Line 5: Other minor problems from when Alex was trying to get things working.
Line 6: "
Line 7: benfulcher_hctsa;0010375160c8434599743992b75b32818a358887;Ben Fulcher <ben.d.fulcher@gmail.com>;1440564091;"Added fast implementation of visibility graph (EZ_VisibilityGraph), based on existing code, and implemented new visibility graph methods into INP_ops/INP_mops
Line 8: "
Line 9: benfulcher_hctsa;00311d91f87cc13464c76e2885ea1b26215f9ffb;GitHub Action <action@github.com>;1738278259;"ghrs: vc agg 01-30-2304-ED44 for benfulcher/hctsa
Line 10: "


In [5]:

import pandas as pd

# Convert Unix timestamps to UTC datetimes
summary_df["date"] = pd.to_datetime(
    summary_df["time"], unit="s", utc=True
)

stats = []

for project_id, project_df in summary_df.groupby("project_wocid"):
    project_df = project_df.sort_values("date")

    # Monthly commit counts; missing months become zero
    monthly = (
        project_df.set_index("date")
        .resample("MS")
        .size()
    )

    # Find all consecutive zero-commit periods
    gaps = []
    gap_start = None

    for month, count in monthly.items():
        if count == 0 and gap_start is None:
            gap_start = month
        elif count > 0 and gap_start is not None:
            gap_end = month - pd.offsets.MonthBegin(1)
            gap_length = len(
                pd.date_range(gap_start, gap_end, freq="MS")
            )
            gaps.append((gap_start, gap_end, gap_length))
            gap_start = None

    # Handle a gap at the end of the monthly timeline
    if gap_start is not None:
        gap_end = monthly.index[-1]
        gap_length = len(
            pd.date_range(gap_start, gap_end, freq="MS")
        )
        gaps.append((gap_start, gap_end, gap_length))

    # Longest gap; ties use the earliest occurrence
    if gaps:
        longest = max(gaps, key=lambda gap: gap[2])
        start, end, length = longest

        # Commits in months strictly after the longest gap
        commits_after_gap = int(
            (
                project_df["date"].dt.to_period("M")
                > end.to_period("M")
            ).sum()
        )

        start_str = start.strftime("%Y-%m")
        end_str = end.strftime("%Y-%m")
    else:
        start_str = ""
        end_str = ""
        length = 0
        commits_after_gap = len(project_df)

    stats.append({
        "project_wocid": project_id,
        "number_of_commits": len(project_df),
        "number_of_authors": project_df["author"].nunique(),
        "max_time": int(project_df["time"].max()),
        "min_time": int(project_df["time"].min()),
        "LongestGapStart (YYYY-MM)": start_str,
        "LongestGapEnd (YYYY-MM)": end_str,
        "LongestGapLength (months)": length,
        "NumCommitsAfterLastGap": commits_after_gap,
        "NumberOfGapsInTimeline": sum(
            gap[2] >= 3 for gap in gaps
        ),
        "ActivityPattern": ""
    })

woc_stats = pd.DataFrame(stats)

# Add readable UTC dates
woc_stats["max_time_utc"] = pd.to_datetime(
    woc_stats["max_time"], unit="s", utc=True
)
woc_stats["min_time_utc"] = pd.to_datetime(
    woc_stats["min_time"], unit="s", utc=True
)

# Merge GitHub stars, forks, last commit date, etc.
part2_report = woc_stats.merge(
    github_df,
    on="project_wocid",
    how="left"
)

# Save the report
part2_report.to_csv(
    f"{NETID}_project_stats.csv",
    index=False,
    sep=";"
)

display(part2_report)


NameError: name 'summary_df' is not defined

In [2]:

import re
import pandas as pd
from collections import Counter

# --------------------------------------------------
# Configuration
# --------------------------------------------------
NETID = "aberard"

commits_output = f"{NETID}_project_gap_commits.csv"
stats_output = f"{NETID}_project_stats.csv"

# Work on a copy of the original commit-level data.
commits = summary_df.copy()
commits["date"] = pd.to_datetime(
    commits["time"], unit="s", utc=True
)
commits["month"] = commits["date"].dt.to_period("M")

# Work on a copy of the existing project statistics.
stats = part2_report.copy()

# --------------------------------------------------
# Classify commit messages into themes
# --------------------------------------------------
def classify_theme(row):
    message = str(row["commit message"]).lower()
    author = str(row["author"]).lower()
    text = message + " " + author

    # Check more specific categories before general ones.
    if any(x in text for x in [
        "dependabot", "renovate", "github action",
        "github-actions", "bot@", "[bot]", "automated"
    ]):
        return "Automated bot contributions"

    if any(x in text for x in [
        "security", "vulnerability", "cve-", "exploit",
        "xss", "injection", "authentication bypass"
    ]):
        return "Security patches"

    if any(x in text for x in [
        "release", "version bump", "version:", "changelog",
        "prepare version", "tagging"
    ]):
        return "Release"

    if any(x in text for x in [
        "dependency", "dependencies", "upgrade package",
        "update package", "bump ", "requirements.txt",
        "lockfile", "package-lock"
    ]):
        return "Dependency updates"

    if any(x in text for x in [
        "documentation", "docs:", "readme", "typo",
        "docstring", "api docs"
    ]):
        return "Documentation updates"

    if any(x in text for x in [
        "bug", "fix", "patch", "crash", "error",
        "incorrect", "regression", "issue"
    ]):
        return "Bug fixes"

    if any(x in text for x in [
        "add ", "added ", "implement", "feature",
        "introduce", "support for", "new "
    ]):
        return "Feature development"

    return "Other"


commits["Theme"] = commits.apply(classify_theme, axis=1)

# --------------------------------------------------
# Select commits immediately before and after each gap
# --------------------------------------------------
gap_commit_frames = []
theme_results = []

for _, project_stats in stats.iterrows():
    project_id = project_stats["project_wocid"]

    project_commits = (
        commits[commits["project_wocid"] == project_id]
        .sort_values("date")
        .copy()
    )

    if project_commits.empty:
        continue

    gap_length = int(
        project_stats["LongestGapLength (months)"]
    )

    start_value = project_stats["LongestGapStart (YYYY-MM)"]
    end_value = project_stats["LongestGapEnd (YYYY-MM)"]

    if gap_length > 0 and pd.notna(start_value) and pd.notna(end_value):
        # The actual longest zero-commit interval.
        gap_start = pd.Period(str(start_value), freq="M")
        gap_end = pd.Period(str(end_value), freq="M")

        # Select the ten commits closest to the gap on each side.
        before = project_commits[
            project_commits["month"] < gap_start
        ].tail(10)

        after = project_commits[
            project_commits["month"] > gap_end
        ].head(10)

    else:
        # Fallback for projects with no zero-commit months:
        # split the observed timeline at its midpoint.
        # There is no actual inactivity gap for these projects.
        first_month = project_commits["month"].min()
        last_month = project_commits["month"].max()
        midpoint = first_month + (
            last_month.ordinal - first_month.ordinal
        ) // 2

        before = project_commits[
            project_commits["month"] <= midpoint
        ].tail(10)

        after = project_commits[
            project_commits["month"] > midpoint
        ].head(10)

    # Add the pre/post label as the first column.
    before_export = before.drop(
        columns=["date", "month", "Theme"]
    ).copy()
    before_export.insert(0, "pre/post", "pre")

    after_export = after.drop(
        columns=["date", "month", "Theme"]
    ).copy()
    after_export.insert(0, "pre/post", "post")

    gap_commit_frames.extend([before_export, after_export])

    # Determine the two most common themes in each group.
    def top_two_themes(selected):
        if selected.empty:
            return ""

        counts = Counter(selected["Theme"])
        top = counts.most_common(2)
        return ":".join(theme for theme, count in top)

    before_themes = top_two_themes(before)
    after_themes = top_two_themes(after)

    theme_results.append({
        "project_wocid": project_id,
        "BeforeThemes": before_themes,
        "AfterThemes": after_themes
    })

# --------------------------------------------------
# Save the selected commit records
# --------------------------------------------------
if gap_commit_frames:
    gap_commits_df = pd.concat(
        gap_commit_frames, ignore_index=True
    )

    gap_commits_df.to_csv(
        commits_output, index=False, sep=";"
    )

    print(f"Saved commit samples to {commits_output}")
    print(f"Total selected commits: {len(gap_commits_df)}")
else:
    print("No commit samples were selected.")

# --------------------------------------------------
# Add BeforeThemes and AfterThemes to project stats
# --------------------------------------------------
themes_df = pd.DataFrame(theme_results)

# Replace old theme columns if rerunning this cell.
stats = stats.drop(
    columns=["BeforeThemes", "AfterThemes"],
    errors="ignore"
)

stats = stats.merge(
    themes_df,
    on="project_wocid",
    how="left",
    validate="one_to_one"
)

stats.to_csv(stats_output, index=False, sep=";")

print(f"Saved updated statistics to {stats_output}")
display(stats[[
    "project_wocid",
    "LongestGapStart (YYYY-MM)",
    "LongestGapEnd (YYYY-MM)",
    "LongestGapLength (months)",
    "BeforeThemes",
    "AfterThemes"
]])


NameError: name 'summary_df' is not defined